# Сравнение моделей прогнозирования

**Цель:** Сравнить ARIMA, ETS, Prophet, XGBoost и базовую модель (среднее) на 4 SKU с использованием скользящего/расширяющегося окна.

**Вход:** `datasetname.xlsx` (не приложен).

**Основные шаги:** генерация признаков, создание лагов, TimeSeriesSplit, обучение моделей, оценка MAE/RMSE.

**Навыки:** statsmodels, pmdarima, Prophet, XGBoost, scikit-learn, валидация временных рядов.

In [ ]:
# Импортируем библиотеки
import pandas as pd
import numpy as np
from pmdarima import auto_arima
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from datetime import timedelta
from collections import defaultdict
from prophet import Prophet
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_excel(datasetname)
df = df.sort_values('Дата').reset_index(drop=True)
df['Месяц'] = df['Дата'].dt.month
lags = [2, 3] # Создаем лаги 2 и 3 порядка
for lag in lags:
    df[f'начальный_остаток_шт_lag{lag}'] = df['Начальный_остаток_шт'].shift(lag)
    df[f'приход_шт_lag{lag}'] = df['Приход_шт'].shift(lag)
    df[f'расход_шт_lag{lag}'] = df['Расход_шт'].shift(lag)
    df[f'выручка_lag{lag}'] = df['Выручка'].shift(lag)
    df[f'приход_lag{lag}'] = df['Приход'].shift(lag)
    df[f'расход_lag{lag}'] = df['Расход'].shift(lag)
    df[f'конечный_остаток_lag{lag}'] = df['Конечный_остаток'].shift(lag)
    df[f'начальный_остаток_lag{lag}'] = df['Начальный_остаток'].shift(lag)  
# Удаляем первые строки с NaN от лагов
df = df.dropna().reset_index(drop=True)

In [ ]:
series = df.set_index('Дата')['Расход_шт']
series.index.freq = 'MS'
start_train = pd.to_datetime('2021-04-01') #из-за лагов 2 и 3
end_test = pd.to_datetime('2024-12-01')
test_dates = pd.date_range(start='2024-01-01', end=end_test, freq='MS')
features = ['начальный_остаток_шт_lag2', 'приход_шт_lag2', 'расход_шт_lag2', 'выручка_lag2',
    'приход_lag2', 'расход_lag2', 'конечный_остаток_lag2', 'начальный_остаток_lag2',
    'начальный_остаток_шт_lag3', 'приход_шт_lag3', 'расход_шт_lag3', 'выручка_lag3',
    'приход_lag3', 'расход_lag3', 'конечный_остаток_lag3', 'начальный_остаток_lag3',
    'Категория', 'Год', 'СКЛАД_запас', 'Месяц', 'Квартал']
# Словари для результатов
results = defaultdict(list)
actuals = []

In [ ]:
param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2], 'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]}
tscv = TimeSeriesSplit(n_splits=3)
xgb = XGBRegressor(random_state=88)
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid,
                           cv=tscv, n_jobs=-1
                           scoring='neg_mean_absolute_error')
# Подбор на начальной части данных
X_initial = df[df['Дата'] < '2024-01-01'][features]
y_initial = df[df['Дата'] < '2024-01-01']['Расход_шт']
grid_search.fit(X_initial, y_initial)
best_params = grid_search.best_params_

In [ ]:
predictions_arima = []
predictions_xgb = []
predictions_baseline = []
predictions_prophet = []
predictions_ets = []
# Цикл скользящего расширяющегося окна
for test_point in test_dates:
    train_end = test_point - pd.DateOffset(months=1)

    # Временной ряд для ARIMA и ETS
    y_train_arima = series[start_train:train_end]
    if test_point not in series.index:
        continue
    y_test_val = series.loc[test_point]
    actuals.append(y_test_val)

    # ARIMA
    model_arima = auto_arima(y_train_arima, seasonal=True, 
                             m=12, stepwise=True, 
                             suppress_warnings=True, error_action='ignore')
    forecast_arima = model_arima.predict(n_periods=1).iloc[0]
    results['ARIMA'].append(forecast_arima)
    predictions_arima.append((test_point, forecast_arima, y_test_val))

    # ETS
    model_ets = ExponentialSmoothing(y_train_arima, 
                                     trend='add', 
                                     seasonal='add', 
                                     seasonal_periods=12)
    model_ets_fit = model_ets.fit(optimized=True)
    forecast_ets = model_ets_fit.forecast(1).iloc[0]
    results['ETS'].append(forecast_ets)
    predictions_ets.append((test_point, forecast_ets, y_test_val))

    # XGBoost
    train_df = df[(df['Дата'] >= start_train) & (df['Дата'] <= train_end)]
    test_df = df[df['Дата'] == test_point]

    if not test_df.empty:
        X_train = train_df[features]
        y_train = train_df['Расход_шт']
        X_test = test_df[features]

        model_xgb = XGBRegressor(**best_params, random_state=88)
        model_xgb.fit(X_train, y_train)
        forecast_xgb = model_xgb.predict(X_test)[0]
        results['XGBoost'].append(forecast_xgb)
        predictions_xgb.append((test_point, forecast_xgb, y_test_val))

        # Baseline (среднее значение по train)
        baseline_forecast = y_train_arima.mean()
        results['Среднее'].append(baseline_forecast)
        predictions_baseline.append((test_point, baseline_forecast, y_test_val))

    # Prophet
    train_prophet = pd.DataFrame({
        'ds': df[(df['Дата'] >= start_train) & (df['Дата'] <= train_end)]['Дата'],
        'y': df[(df['Дата'] >= start_train) & (df['Дата'] <= train_end)]['Расход_шт']
    })
    model_prophet = Prophet()
    model_prophet.fit(train_prophet)

    future = model_prophet.make_future_dataframe(periods=1, freq='MS')
    forecast_prophet = model_prophet.predict(future).tail(1)['yhat'].values[0]
    results['Prophet'].append(forecast_prophet)
    predictions_prophet.append((test_point, forecast_prophet, y_test_val))

# Оценка MAE
print("MAE ARIMA:", mean_absolute_error(actuals, results['ARIMA']))
print("MAE XGBoost:", mean_absolute_error(actuals, results['XGBoost']))
print("MAE Среднее:", mean_absolute_error(actuals, results['Среднее']))
print("MAE Prophet:", mean_absolute_error(actuals, results['Prophet']))
print("MAE ETS:", mean_absolute_error(actuals, results['ETS']))

In [ ]:
final_results = pd.DataFrame({
    'Дата': test_dates,
    'Фактические': actuals,
    'ARIMA': results['ARIMA'],
    'XGBoost': results['XGBoost'],
    'Среднее': results['Среднее'],
    'Prophet': results['Prophet'],
    'ETS': results['ETS']
})

# Добавим строки с MAE и RMSE
metric_rows = []

# MAE строка
mae_row = {'Дата': 'MAE', 'Фактические': ''}
for model in results:
    mae_row[model] = round(mean_absolute_error(actuals, results[model]), 2)
metric_rows.append(mae_row)

# RMSE строка
rmse_row = {'Дата': 'RMSE', 'Фактические': ''}
for model in results:
    rmse_row[model] = round(root_mean_squared_error(actuals, results[model]), 2)
metric_rows.append(rmse_row)

# Объединяем всё
final_results = pd.concat([final_results, pd.DataFrame(metric_rows)], ignore_index=True)

In [ ]:
final_results = final_results.rename(columns={'Фактические': 'Факт'})
new_column_order = ['Дата', 'Факт', 'Среднее', 'ARIMA', 'XGBoost', 'Prophet', 'ETS']
final_results = final_results[new_column_order]
# Сохранение в файл
filename = f'{num}_NEW_models_{version}.xlsx'
final_results.to_excel(filename, index=False)

In [ ]:
# Построение графика для сравнения моделей
plt.figure(figsize=(10, 5))
plt.plot(actuals, label='Факт', marker='o')
for model in results:
    plt.plot(results[model], label=model, marker='x')
plt.title(f'Сравнение моделей на тестовой выборке товар №{num}')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(alpha=0.5)
plt.tight_layout() 
plt.show()